# Gradientes, control de compresión y figuras (OE2)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import subprocess

from google.colab import userdata

REPO = "github.com/Riv2oo4/Trabajo-De-Graduacion.git"
RAMA = "oe4-oe5-y-reproducibilidad"

token = userdata.get("GITHUB_TOKEN")

# si una corrida anterior dejó el proceso parado dentro de /content/repo,
# borrarlo deja getcwd sin directorio y ningún subproceso vuelve a arrancar
os.chdir("/content")

!rm -rf /content/repo
subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                f"https://x-access-token:{token}@{REPO}", "/content/repo"], check=True)

assert os.path.exists("/content/repo/analisis/gradientes.py"), \
    f"el clon de {RAMA} no trae gradientes.py: revisar que el clon no haya fallado"
%cd /content/repo/analisis

In [ ]:
!pip install -q scipy matplotlib

## Datos

In [ ]:
DRIVE = "/content/drive/MyDrive/Corpus-tesis"
ZIP_REALES = f"{DRIVE}/corpus_real.zip"
MANIPULADAS = f"{DRIVE}/manipuladas"
RESULTADOS = f"{DRIVE}/resultados"

assert os.path.exists(ZIP_REALES), ZIP_REALES
assert os.path.isdir(MANIPULADAS), MANIPULADAS
os.makedirs(RESULTADOS, exist_ok=True)

!mkdir -p /content/reales && unzip -q -o "{ZIP_REALES}" -d /content/reales

REALES = "/content/reales/real"
CATALOGO = "/content/repo/corpus/catalogo.csv"
print(len(os.listdir(REALES)), "reales |", len(os.listdir(MANIPULADAS)), "manipuladas")

## Auditoría comparativa de las dos clases

In [ ]:
import csv
import shutil

from gradientes import region_de

# la auditoría caracteriza una carpeta a partir de su metadata; se arma una por
# clase desde el catálogo para poder compararlas entre sí
filas = list(csv.DictReader(open(CATALOGO)))
for clase, origen in (("real", REALES), ("mani", MANIPULADAS)):
    destino = f"/content/auditoria_{clase}"
    os.makedirs(destino, exist_ok=True)
    with open(f"{destino}/metadata.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["archivo", "region"])
        w.writeheader()
        for f in filas:
            w.writerow({"archivo": f["archivo"],
                        "region": region_de(f["archivo"])})
    for f in filas:
        ruta = f"{origen}/{f['archivo']}"
        if os.path.exists(ruta) and not os.path.exists(f"{destino}/{f['archivo']}"):
            os.symlink(ruta, f"{destino}/{f['archivo']}")
print("carpetas de auditoría listas")

In [ ]:
# el criterio es que las distribuciones queden indistinguibles: cualquier
# diferencia sistemática de resolución, proporción, formato o calidad sería
# aprendible por el detector en lugar de la manipulación
print("=" * 70, "\nCLASE AUTÉNTICA\n", "=" * 70)
!python auditoria_corpus.py --dataset /content/auditoria_real
print("=" * 70, "\nCLASE MANIPULADA\n", "=" * 70)
!python auditoria_corpus.py --dataset /content/auditoria_mani

## Medición de gradientes

In [ ]:
CMD = (
    "python gradientes.py"
    f" --catalogo {CATALOGO}"
    f" --reales {REALES}"
    f" --manipuladas '{MANIPULADAS}'"
    f" --salida '{RESULTADOS}/gradientes.csv'"
    f" --contraste '{RESULTADOS}/gradientes_contraste.csv'"
)
!{CMD}

## Control de compresión

In [ ]:
# aplica a la auténtica la pasada JPEG adicional que su pareja ya acumula; si
# el efecto sobrevive, no procede del recodificado
CMD = (
    "python gradientes.py"
    f" --catalogo {CATALOGO}"
    f" --reales {REALES}"
    f" --manipuladas '{MANIPULADAS}'"
    " --pasada recompresion"
    f" --salida '{RESULTADOS}/gradientes_recompresion.csv'"
    f" --contraste '{RESULTADOS}/gradientes_contraste_recompresion.csv'"
)
!{CMD}

## Baseline del OE3

In [ ]:
!python baseline_umbral.py --csv "{RESULTADOS}/gradientes.csv" --salida "{RESULTADOS}/baseline_umbral.csv"

## Figuras

In [ ]:
CMD = (
    "python figuras_gradiente.py"
    f" --catalogo {CATALOGO}"
    f" --reales {REALES}"
    f" --manipuladas '{MANIPULADAS}'"
    f" --contraste '{RESULTADOS}/gradientes_contraste.csv'"
    f" --salida '{RESULTADOS}/figuras'"
    " --cuantas 20"
)
!{CMD}

## Resumen del contraste

In [ ]:
import pandas as pd

contraste = pd.read_csv(f"{RESULTADOS}/gradientes_contraste.csv")
dentro = contraste[contraste.zona.eq("mascara") & contraste.grupo.eq("todos")]
dentro[["operador", "metrica", "n", "mediana_real", "mediana_mani",
        "cambio_relativo", "p", "r", "sentido"]]

In [ ]:
# desagregado por nivel, por región y por referente
contraste[contraste.zona.eq("mascara") & contraste.metrica.eq("magnitud")
          & contraste.grupo.ne("todos")][
    ["grupo", "operador", "n", "cambio_relativo", "p", "r"]]

In [ ]:
# el control: mismas filas con la auténtica recomprimida
ctrl = pd.read_csv(f"{RESULTADOS}/gradientes_contraste_recompresion.csv")
comparacion = dentro.merge(
    ctrl[ctrl.zona.eq("mascara") & ctrl.grupo.eq("todos")],
    on=["operador", "metrica", "zona", "grupo"], suffixes=("_base", "_ctrl"))
comparacion[["operador", "metrica", "r_base", "r_ctrl",
             "cambio_relativo_base", "cambio_relativo_ctrl"]]

In [ ]:
from IPython.display import Image as Ver

Ver(filename=f"{RESULTADOS}/figuras/resumen_efecto.png")